In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from tqdm import tqdm

In [ ]:
# Load data
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
# Check the structure and columns
print(data.head())
print(data.columns)

In [ ]:
# Calculate correlation between specific columns
print(data[['mvel1', 'mve_ia', 'rd_mve']].corr())

In [ ]:
# # Get the latest date records and sort by market value
# latest_dates = data.groupby('permno')['DATE'].max().reset_index()
# latest_data = pd.merge(data, latest_dates, on=['permno', 'DATE'], how='inner')
# top_1000 = latest_data.sort_values(by='mvel1', ascending=False)['permno'].values[:1000]
# last_1000 = latest_data.sort_values(by='mvel1', ascending=False)['permno'].values[-1001:-1]

In [ ]:
# # Filter data for top 1000 and last 1000 companies
# data_top_1000 = data[data['permno'].isin(top_1000)].reset_index(drop=True)
# data_last_1000 = data[data['permno'].isin(last_1000)].reset_index(drop=True)

In [ ]:
# Define R² calculation function
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y - y_pred) ** 2) / np.sum(y ** 2)
    return r2

In [ ]:
# Define function to train and test the model using PCR
def train_pcr(data, n_components_list=[2, 5, 10, 15, 20]):
    start_year_train = 1957
    start_year_test = 1987
    end_year_test = 2016
    
    data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    pred_y_list = []
    true_y_list = []

    # Standardize the features
    scaler = StandardScaler()

    for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
        start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
        end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)
        start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
        end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)
        start_date_test = pd.Timestamp(year=year, month=1, day=1)
        end_date_test = pd.Timestamp(year=year, month=12, day=31)
        
        train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
        test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
        
        X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
        y_train = train_data['RET'].values  
        X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_test = test_data['RET'].values
        
        # Standardize the features
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # Use cross-validation to find the best number of components
        best_n_components = n_components_list[0]
        best_score = -np.inf

        for n_components in n_components_list:
            pca = PCA(n_components=n_components)
            X_train_pca = pca.fit_transform(X_train)
            model = LinearRegression()
            
            # Perform cross-validation
            cv = KFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(model, X_train_pca, y_train, cv=cv, scoring='r2')
            mean_score = np.mean(scores)

            if mean_score > best_score:
                best_score = mean_score
                best_n_components = n_components
        
        # Train the final model with the best number of components
        pca = PCA(n_components=best_n_components)
        X_train_pca = pca.fit_transform(X_train)
        X_test_pca = pca.transform(X_test)
        
        model = LinearRegression()
        model.fit(X_train_pca, y_train)
        y_test_pred = model.predict(X_test_pca)
        
        pred_y_list.extend(y_test_pred)
        true_y_list.extend(y_test)
         # Save results to CSV file
        results_df = pd.DataFrame({
            'Date': test_data['DATE'],
            'True_RET': y_test,
            'Pred_RET': y_test_pred
        })
        results_df.to_csv(f'PCR/{year}.csv', index=False)
    
    r2_result = R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100
    print('R²:{}'.format(r2_result))
    return r2_result

In [ ]:
# Train and test models
r2_full = train_pcr(data)